# Bloque 4 · Elementos de prueba — prácticas en Google Colab
## Del dato a la evidencia verificable

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/consumer-spain/curso-ia-colab/blob/main/bloque4/bloque4_practicas.ipynb)

**Esto es el Plan B.** Si en tu ordenador algo no funciona (Python, paquetes, permisos…), puedes seguir aquí
**todas las prácticas del Bloque 4**, en el mismo orden que en clase.

**Cómo se usa**
- Cada bloque gris es una *celda*. Se ejecuta con el botón ▶ de su izquierda (o **Mayús + Enter**).
- Ejecuta las celdas **en orden**, de arriba abajo.
- Si Colab se desconecta o lo reinicias, vuelve a ejecutar la celda **Preparación**.
- Colab **no guarda tus cambios**. Si quieres conservarlos: *Archivo → Guardar una copia en Drive*.
- No subas aquí datos reales ni información interna: todo el material del curso es de prueba.

| En clase | Aquí |
|---|---|
| Doble clic en un `.bat` o arrastrar archivos encima | Ejecutar una celda (`!python archivo.py …`) |
| Explorador: copiar, pegar y renombrar (Ctrl+C, Ctrl+V, F2) | Una celda que hace lo mismo con Python |
| Bloc de notas, Paint, Excel | Una celda que muestra el archivo (`!cat`, `display`, pandas) o que lo modifica |
| La ventana del sellador (`tkinter`) | Un formulario de Colab y una celda que hace de botón |
| `/exif`, `/hash`, `/ip`, `/verificador` en la web del curso | **Igual**, en tu navegador. Los archivos se descargan desde Colab |

> **Importante.** Lo que se obtiene desde Colab sale de servidores de Google, no de un equipo institucional:
> **no tiene valor probatorio**. Es una réplica de las prácticas para aprender, no una forma de producir evidencia.

## Preparación

Descarga los materiales del Bloque 4 (el mismo ZIP que en clase), instala lo que falte y ejecuta el comprobador.
Es el equivalente a `COMPROBAR_BLOQUE4.bat`. Debe terminar en **TODO LISTO** o **HAY ALGO QUE REVISAR**.

*Adaptada.* En Colab no hay ventanas: el comprobador avisará de que falta `tkinter`; es lo esperable, y el sellador
se usa por celdas. También hace una prueba de sellado con la TSA: puede tardar hasta 10 segundos.

In [ ]:
# PREPARACIÓN · ejecuta esta celda primero
import os, sys, zipfile, urllib.request, subprocess
from pathlib import Path

ZIP_URL = "https://raw.githubusercontent.com/consumer-spain/curso-ia-colab/main/bloque4/BLOQUE4_ALUMNADO.zip"
TRABAJO = Path.cwd().resolve()
BASE = TRABAJO / "BLOQUE4"

if not BASE.exists():
    print("Descargando los materiales del Bloque 4...")
    urllib.request.urlretrieve(ZIP_URL, TRABAJO / "materiales_b4.zip")
    with zipfile.ZipFile(TRABAJO / "materiales_b4.zip") as z:
        z.extractall(TRABAJO)
    print("Materiales listos en:", BASE)
else:
    print("Los materiales ya estaban descargados:", BASE)

# Paquetes (normalmente Colab ya los trae)
for modulo, paquete in (("pandas", "pandas"), ("openpyxl", "openpyxl"), ("requests", "requests"),
                        ("bs4", "beautifulsoup4"), ("PIL", "Pillow")):
    try:
        __import__(modulo)
    except ImportError:
        print("Instalando", paquete, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import pandas as pd
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 60)
from IPython.display import Image as VerImagen, display


def ir_a(carpeta):
    """Se coloca en una carpeta del material (como abrir la carpeta en el Explorador)."""
    os.chdir(BASE / carpeta if carpeta else BASE)
    print("Estás en:", Path.cwd().relative_to(TRABAJO))


def ver_excel(ruta, hoja=0):
    """Muestra un Excel (en clase se abre con Excel)."""
    return pd.read_excel(ruta, sheet_name=hoja)


def descargar(*rutas):
    """Descarga archivos a tu ordenador (para usarlos en la web del curso o en tu IA)."""
    try:
        from google.colab import files
        for r in rutas:
            files.download(str(r))
    except ImportError:
        print("Esta celda solo descarga archivos dentro de Google Colab.")


def ultimo_paquete():
    """Devuelve la carpeta EVIDENCIA_… más reciente creada por tu sellador (o None)."""
    base = BASE / "03_SELLADOR" / "MIS_PAQUETES"
    if not base.exists():
        return None
    paquetes = sorted(base.glob("EVIDENCIA_*"), key=lambda p: p.stat().st_mtime)
    return paquetes[-1] if paquetes else None


ir_a("")
print()
r = subprocess.run([sys.executable, "comprobar_bloque4.py"], capture_output=True, text=True)
import re
print(re.split(r"\n(?=HAY ALGO QUE REVISAR|TODO LISTO)", r.stdout)[0])
problemas = [l for l in r.stdout.splitlines() if l.startswith("  - ")]
otros = [l for l in problemas if "tkinter" not in l]
TSA_OK = "sello real recibido" in r.stdout
if r.returncode == 0:
    print("TODO LISTO")
elif not otros:
    print("TODO LISTO (en Colab no hay ventanas: la falta de tkinter es esperable; el sellador se usa por celdas)")
else:
    print("HAY ALGO QUE REVISAR")
    for l in otros:
        print(l)
if not TSA_OK:
    print("\nNota: la TSA no ha respondido desde Colab. Es posible que sea un bloqueo de la red de Google;")
    print("el taller del sellador lo tiene previsto (paquete de contingencia).")

---
# A · Apertura

**Muchas veces, el dato basta.** Estudios de mercado, monitorización, estadística, seguimiento de precios, indicios.
**A veces necesitamos algo más:** poder acreditarlo después.

### La pregunta de hoy
**¿Cómo pasamos de un dato útil a una evidencia que pueda comprobarse después?**

### Un método en seis pasos
**RECOGER → CONTEXTUALIZAR → FIJAR → VINCULAR AL TIEMPO → DOCUMENTAR → VERIFICAR**

Hoy recorremos los seis pasos con ejemplos y herramientas. *No hay nada que ejecutar en esta parte.*

---
# B · Books to Scrape

Partimos de lo que ya sabemos hacer (Bloque 3): obtener una tabla de una web. Aquí, los 20 libros de la primera página
de *Books to Scrape*, una web de pruebas.

> **IGUAL / ADAPTADA.** El programa es el mismo. En clase se lanza con `EJECUTAR_BOOKS.bat`; aquí con una celda.
> Si la web no responde desde los servidores de Google, la celda repite sola con la copia local de respaldo
> (exactamente lo que hace el `.bat`).

## Ejecutar

In [ ]:
ir_a("01_BOOKS")
!python obtener_libros.py || (echo "La web no ha respondido. Repetimos con la copia local de respaldo..." && python obtener_libros.py --respaldo)

## Abrir el Excel

En clase se abre `libros_books.xlsx` con Excel. Aquí lo mostramos con pandas (también puedes descargarlo con el panel 📁).

In [ ]:
ver_excel("libros_books.xlsx")

## ¿Lo daríais por bueno?

Piénsalo antes de seguir: ¿qué has comprobado de esta tabla? ¿Qué no puedes saber solo con mirarla?

### Mini-reto · fila 7 · 3 min · por parejas

Fíjate **solo en la fila 7**. Dentro de seis meses alguien te pregunta: *«¿cómo sabemos de dónde salió esta fila?»*

**Para una sola fila: ¿qué guardaríais hoy?** Apunta todas tus ideas (en una celda de texto, en papel…).

*Después, en común, ordenaremos lo que habéis dicho: la página tal como se vio · la URL, la hora y con qué herramienta ·
algo que avise si el archivo cambia · una hora que no dependa de mi equipo · una nota que explique cada archivo · comprobar que todo encaja.
Lo que habéis propuesto ya es un método.*

---
# C · RECOGER

**Guardar la fila no basta: guardamos lo que vimos.** Recogemos la ficha completa del libro 7, tal como llegó,
más un registro de qué, de dónde, cuándo y cómo.

> **IGUAL.** (`RECOGER_LIBRO_7.bat` → una celda; con el mismo respaldo si no hay web.)

In [ ]:
ir_a("01_BOOKS")
!python recoger_libro.py 7 || (echo "Repetimos con la copia local de respaldo..." && python recoger_libro.py 7 --respaldo)

Mira qué se ha creado y lee el registro. Fíjate en los cuatro bloques: **qué se recogió · de dónde · cuándo · cómo**.

In [ ]:
!ls RECOGIDA_libro_07
!cat RECOGIDA_libro_07/registro_recogida.json

> Ojo con el «cuándo»: es la **hora del reloj de este equipo**. Es una declaración, no una garantía externa.

**Recoger = conservar lo que vimos, no solo el dato que extrajimos.**

## Caso guiado · Orion 3

Para los siguientes pasos trabajamos con un caso: la captura de una oferta de la cafetera *Orion 3*,
hecha con *InspectorApp* (una aplicación ficticia del curso). En clase está en `02_INSPECTORAPP`; aquí la ves directamente.

In [ ]:
ir_a("02_INSPECTORAPP")
display(VerImagen("captura_original.jpg", width=760))

---
# D · CONTEXTUALIZAR

**Un archivo no solo contiene lo que vemos.** Los metadatos dicen cosas del archivo sobre sí mismo.

> **Windows → Propiedades → Detalles** (clic derecho sobre la imagen): **No exportable**; en Colab no hay Explorador.
> Usamos directamente la herramienta del curso, que además compara dos archivos.

## Práctica · metadatos de la original y de la reducida · 3 min

**No leas todo. Busca estos campos:** Software · Dimensiones · Fecha declarada · Dispositivo · GPS.
Primero en `captura_original.jpg`; después compara con `captura_reducida.jpg`.

*(En clase: `1_VER_METADATOS.bat`.)*

In [ ]:
ir_a("02_INSPECTORAPP")
!python ver_metadatos.py captura_original.jpg captura_reducida.jpg

### En la web del curso · `/exif`

*(Igual, en tu navegador.)* Abre `/exif` en la web del curso (la dirección que se proyecta en clase) y sube
`captura_original.jpg` como imagen A y `captura_reducida.jpg` como imagen B. Primero las descargas:

In [ ]:
descargar("captura_original.jpg", "captura_reducida.jpg")

## ¿Qué observas?

- ¿Qué cambia entre las dos? ¿Qué campo sigue siendo el mismo?
- ¿Cuánto puedes fiarte de lo que el propio archivo dice de sí mismo?

*Los metadatos forman parte del archivo: se pueden alterar o desaparecer.* **METADATO ≠ DATO CERTIFICADO.**

### Qué afirmar y qué no
Un metadato permite decir que el archivo **es compatible** con algo, no que ese algo esté demostrado. **COMPATIBLE ≠ DEMOSTRADO.**

## Mini-reto · `/ip` · ¿desde dónde llegas tú? · 2 min

> **IGUAL, y solo en tu navegador.** Abre `/ip` en la web del curso (QR o dirección proyectada).
> La web te muestra la IP desde la que le llega *tu* petición. **No lo hagas desde Colab:** aquí la petición sale de Google, no de ti.

Pensad: ¿cambia si cambias de red (wifi o datos del móvil)? ¿Una IP identifica a una persona? **IP OBSERVADA ≠ PERSONA IDENTIFICADA.**
Una IP sola frente al registro que guarda el servidor… ya lo hemos comentado: la IP dice desde qué red llegó la petición, no quién la hizo.

*Solo como contraste (ADAPTADA), esta celda pregunta qué IP vería un servidor si la petición saliera de este cuaderno.
Fíjate en que no es la tuya, sino la de un centro de datos de Google:*

In [ ]:
import urllib.request
try:
    ip = urllib.request.urlopen("https://api.ipify.org", timeout=8).read().decode()
    print("Un servidor que recibiera una petición desde este cuaderno vería la IP:", ip)
    print("Es una IP de centro de datos, no la de tu equipo.")
except Exception as exc:
    print("No se ha podido consultar la IP:", exc)

---
# E · FIJAR

**Antes de tocar archivos: ver la extensión.** En clase se activa en Windows *Ver → Extensiones de nombre de archivo*.
*(No exportable: en Colab las extensiones se ven siempre en el panel 📁 y en `!ls`.)*

## Crear una huella del archivo

**ARCHIVO → FUNCIÓN HASH → HUELLA.** Existen MD5, SHA-1, SHA-256 y SHA-512; **en este curso usamos SHA-256**.

## Práctica · ¿Cuáles son el mismo archivo? · 4 min

**Primero predice. Después compruébalo.** Vamos a crear una copia de `captura_original.jpg` con otro nombre (`mi_copia.jpg`).
¿Qué archivos quedarán en el mismo grupo?

- `captura_original.jpg`
- `mi_copia.jpg`
- `captura_reducida.jpg`

*A la vista, las tres muestran la misma oferta.* **Escribe tu predicción antes de ejecutar nada.**

*(En clase: Ctrl+C → Ctrl+V → F2. Aquí, esta celda: ADAPTADA.)*

In [ ]:
import shutil
ir_a("02_INSPECTORAPP")
shutil.copy("captura_original.jpg", "mi_copia.jpg")
print("Creada mi_copia.jpg")

Ahora calcula las huellas y agrúpalas (en clase: `2_COMPARAR_HASHES.bat`):

In [ ]:
!python comparar_hashes.py captura_original.jpg captura_reducida.jpg mi_copia.jpg

¿Acertaste? ¿Qué te dice esto sobre «el mismo archivo»?

### En la web del curso · `/hash`
*(Igual, en tu navegador.)* Selecciona los tres archivos en `/hash`. Si los necesitas en tu ordenador:

In [ ]:
descargar("captura_original.jpg", "captura_reducida.jpg", "mi_copia.jpg")

## Mini-reto · Cambia un solo carácter · 2 min

**¿Cuánto cambiará la huella?**

- **A.** cambia un carácter de la huella
- **B.** cambian unos pocos
- **C.** cambia casi todo

Elige antes de ejecutar. Ahora haremos una copia de `ARCHIVO_DE_MUESTRA.txt`, le añadiremos **un punto** al final
y volveremos a calcular la huella. **Se modifica la copia; el original no se toca.**

*(En clase: copiar, Bloc de notas, Ctrl+S, `2_COMPARAR_HASHES.bat`. Aquí: ADAPTADA, con celdas.)*

In [ ]:
import shutil
ir_a("02_INSPECTORAPP")
shutil.copy("ARCHIVO_DE_MUESTRA.txt", "ARCHIVO_DE_MUESTRA - copia.txt")
with open("ARCHIVO_DE_MUESTRA - copia.txt", "a", encoding="utf-8") as f:
    f.write(".")            # un único carácter más
print("Copia modificada: se ha añadido un punto al final.")

In [ ]:
!python comparar_hashes.py "ARCHIVO_DE_MUESTRA.txt" "ARCHIVO_DE_MUESTRA - copia.txt"

Compara las dos huellas, carácter a carácter. ¿Era A, B o C?

## Lo que nos da SHA-256… y lo que no

Nos da una huella que **cambia si cambian los bytes**. No nos dice si el contenido es verdadero.
**INTEGRIDAD ≠ VERACIDAD.**

---
# F · VINCULAR AL TIEMPO

La huella dice *qué* archivo. No dice *cuándo*. Y las tres fechas que tenemos bajo nuestro control
(la del reloj del equipo, la de modificación y la de los metadatos) **se pueden cambiar**.

**Firma, sello y sello de tiempo no son lo mismo.** Un **sello de tiempo** (RFC 3161) lo emite un tercero, una **TSA**:
recibe solo la **huella** del archivo (no el archivo) y devuelve una respuesta firmada con la hora.
Esa respuesta es el `.tsr`; lo que se envía, la petición, es el `.tsq`.

*(Apunte eIDAS: que un sello cumpla RFC 3161 **no** lo hace cualificado; el prestador y el servicio deben estar en la lista de confianza.
Las presunciones del art. 41 aplican a los cualificados.)*

## Una petición real

Veamos qué viajaría a la TSA, **sin enviar nada**: la petición (`.tsq`) contiene la huella SHA-256 del archivo; el archivo no sale.

In [ ]:
ir_a("03_SELLADOR")
!python ver_peticion.py

## Mini-reto · Marca SÍ o NO · 2 min

Tenemos un sello de tiempo de `captura_original.jpg`. Responde antes de seguir:

1. ¿Demuestra **quién hizo** la captura?
2. ¿Demuestra que **el precio era verdadero**?
3. ¿Demuestra que **la captura se hizo exactamente en ese instante**?
4. ¿Demuestra que **esos bytes ya existían** cuando se emitió el sello?
5. Si hoy la huella coincide, ¿**seguimos teniendo los mismos bytes**?

<details><summary>Ver las respuestas (después de contestar)</summary>

1. **NO.** El sello no identifica a nadie.
2. **NO.** El sello no mira el contenido.
3. **NO.** La captura pudo hacerse antes: el sello dice «como muy tarde».
4. **SÍ.** Esos bytes existían, como muy tarde, en ese instante.
5. **SÍ.** Si la huella coincide, el archivo no ha cambiado desde entonces.

**SELLO DE TIEMPO ≠ FECHA DE CREACIÓN.** *El sellado protege hacia adelante; no convierte en verdadera una captura.*
</details>

## Mini-reto · ¿Qué debería comprobar un verificador? · 2 min

Tienes `captura_original.jpg` + `captura_original_demo.tsr`. **Si fueras el verificador, ¿qué mirarías?**
Piénsalo (o dilo en voz alta) **antes de ver ninguna herramienta**.

## Mini-reto · Mismo sello, dos archivos · 3 min

**¿A cuál de los dos archivos corresponde el sello?** Usa el sello de muestra (`SELLO_DEMO/captura_original_demo.tsr`),
con `captura_original.jpg` y con `captura_reducida.jpg`.

> *Sello de muestra · TSA de demostración del curso · NO CUALIFICADA.*

**Opción 1 · en este cuaderno** (en clase: `3_VERIFICAR_SELLO.bat`):

In [ ]:
ir_a("02_INSPECTORAPP")
!python verificar_sello.py

Lee **cada lectura por separado**: respuesta · origen/TSA · tiempo · huella sellada · huella del archivo · correspondencia.
Este programa **no comprueba la firma ni la cadena de confianza** de la TSA.

**Opción 2 · en la web del curso · `/verificador`** *(Igual, en tu navegador. Aquí sí se comprueban la firma y la cadena.)*
Abre `/verificador`, sube `captura_original.jpg` con el sello y repite con `captura_reducida.jpg` y **el mismo sello**.
Descarga primero los archivos:

In [ ]:
ir_a("02_INSPECTORAPP")
descargar("captura_original.jpg", "captura_reducida.jpg", "SELLO_DEMO/captura_original_demo.tsr")

**Las seis lecturas del verificador:** respuesta legible · origen/TSA · hora · huella sellada · huella del archivo · correspondencia.
¿Qué cambia con la reducida? ¿Qué sigue igual?

## El momento de fijación

Lo que se sella es la **huella del archivo tal como existe en ese instante**. Si se sella una captura ya manipulada,
el sello certifica esa manipulación. Por eso el sellado debe ocurrir **en el momento de fijar**, no después.

---
# G · DOCUMENTAR

## Una carpeta, seis meses después

Tienes una carpeta con una imagen, una petición, un sello… ¿Qué es cada cosa? ¿Cuál es el original, cuál una copia?
**Documentar no es un formato**: es que otra persona pueda entender qué hay y de dónde sale.

### Mini-reto · Original, copia, derivado o modificado · 2 min

Para cada archivo con el que has trabajado (`captura_original.jpg`, `mi_copia.jpg`, `captura_reducida.jpg`, la copia con un punto):
¿es **original**, **copia**, **derivado** o **modificado**? Cada archivo, su papel.

## Una opción: un manifiesto

Un **manifiesto** (`manifiesto.json`) relaciona las piezas y documenta de dónde sale cada dato.
**El manifiesto documenta; no certifica por sí mismo.**

## Mini-reto · Localiza estos campos · 2 min

*No hace falta saber JSON.* Abre el manifiesto del paquete de muestra (en clase: `03_SELLADOR → PAQUETE_CONTINGENCIA →
EVIDENCIA_captura_original_DEMO`, clic derecho → Abrir con → Bloc de notas; aquí, `!cat`).

Localiza **solo** estos campos y di **de qué grupo** son (herramienta · sello · proceso · contexto):
`sha256` · `hora_del_sello` · `huella_sellada` · `tsa_llamada` · `fuente_o_url` · `huella_sellada_corresponde`

In [ ]:
ir_a("03_SELLADOR/PAQUETE_CONTINGENCIA/EVIDENCIA_captura_original_DEMO")
!ls
!cat manifiesto.json

<details><summary>Ver el reparto por procedencia (después de localizarlos)</summary>

- **Calculado por la herramienta:** `sha256`, `huella_sellada_corresponde`
- **Obtenido del sello:** `hora_del_sello`, `huella_sellada`
- **Configuración del proceso:** `tsa_llamada`
- **Aportado como contexto:** `fuente_o_url`

*(Este paquete de contingencia es una muestra técnica con una TSA de demostración no cualificada.)*
</details>

---
# H · VERIFICAR (suficiencia)

## Mini-reto · Dos paquetes técnicamente correctos · 1 min

Supuesto para el aula: el encargo era **acreditar el precio de la cafetera Orion 3**.

| | Huella | Sello | Manifiesto |
|---|---|---|---|
| **Paquete A** | ✓ | ✓ corresponde | ✓ |
| **Paquete B** | ✓ | ✓ corresponde | ✓ |

**¿Los dos sirven?** Piénsalo antes de seguir.

<details><summary>Ver la respuesta</summary>

- **A · técnicamente correcto, insuficiente:** no se ve el precio, que era lo que había que acreditar. Tampoco el pie de InspectorApp.
- **B · conjunto más robusto:** producto, referencia, precio, tienda, URL y aplicación en la misma captura, con su contexto.

**VALIDEZ TÉCNICA ≠ SUFICIENCIA DE LA EVIDENCIA.**
</details>

### La ruta puede volver

Si al verificar vemos que no basta, **volvemos atrás**: VERIFICAR ↺ RECOGER (o ↺ DOCUMENTAR). La ruta no es una línea recta.

---
# I · Taller de integración: el sellador

*Hay cosas que merece la pena hacer bien una vez. Después nos las llevamos.*

## Qué queremos construir

Una herramienta que, dado un archivo, cree un **paquete de evidencia** (`EVIDENCIA_<nombre>`):
copia exacta del original · huella · petición (`.tsq`) · sello (`.tsr`) · `manifiesto.json`.

## Qué tiene que ocurrir por dentro

1. Conservar una copia exacta del archivo · 2. Calcular su huella · 3. Preparar la petición · 4. Pedir el sello a la TSA ·
5. Comprobar que la huella sellada corresponde · 6. Escribir el manifiesto.

## El núcleo que os lleváis: `sellado_core.py`

Ya está hecho y probado. **No se modifica.** Esta es su función principal:

In [ ]:
ir_a("03_SELLADOR")
import sys
sys.path.insert(0, str(Path.cwd()))
import sellado_core
help(sellado_core.sellar_archivo)

## La ventana: lo que falta

En clase, el núcleo ya está encapsulado en una **ventana** con campos y botones (`02_reto_ventana_incompleta.py`);
solo falta que el botón **«CREAR PAQUETE»** haga algo: la función `ejecutar_sellado()`.

> **ADAPTADA.** `tkinter` no abre ventanas en Colab. Hacemos lo mismo con dos piezas:
> un **formulario** de Colab (los campos: archivo, destino, fuente, observaciones) y **una celda que hace de botón**.
> Lo que hay que completar es la misma función, `ejecutar_sellado()`; solo cambian los mensajes (`print` en lugar de ventanas emergentes).

### ¿Qué significa que funcione?
Piénsalo antes de seguir: ¿cómo sabrás que tu herramienta funciona? «Que termine sin error» **no** es suficiente.

### Nuestros criterios de aceptación
- El archivo de partida **no cambia**.
- **No se sobrescribe** un paquete anterior sin que tú lo decidas.
- Se crea `EVIDENCIA_<nombre>` con original, `peticion.tsq`, `sello.tsr` y `manifiesto.json`.
- El manifiesto recoge **tu fuente y tus observaciones**.
- La huella sellada **corresponde** al original.
- Si algo falla, el mensaje **se entiende** y no queda un paquete a medias (salvo que tú elijas «pendiente»).

### Archivos de partida

Esta celda crea el equivalente a `02_reto_ventana_incompleta.py` para Colab (`02_reto_colab_incompleto.py`) y el formulario.
Son los dos archivos que adjuntarás a tu IA junto con `sellado_core.py`.

In [ ]:
ir_a("03_SELLADOR")

In [ ]:
%%writefile 02_reto_colab_incompleto.py
"""TALLER DEL SELLADOR · versión Colab (incompleta).

Lo que en clase es la ventana, aquí es el formulario (formulario.json) y esta función hace de botón «CREAR PAQUETE».
Completa SOLO la función ejecutar_sellado(). El núcleo (sellado_core.py) no se modifica.
"""
import json
from pathlib import Path

from sellado_core import ErrorTSA, SelladoError, sellar_archivo  # noqa: F401  (los usarás)


def leer_formulario():
    """Devuelve (archivo, destino, fuente, observaciones) tal como están en el formulario."""
    datos = json.loads(Path("formulario.json").read_text(encoding="utf-8"))
    return datos["archivo"], datos["destino"], datos["fuente"], datos["observaciones"]


def ejecutar_sellado():
    # TODO 1 · Leer los datos: leer_formulario() devuelve archivo, destino, fuente y observaciones.
    # TODO 2 · Validar: si falta el archivo o el destino, avisar con print y no seguir.
    # TODO 3 · Llamar al núcleo: sellar_archivo(archivo, destino, fuente=..., observaciones=...)
    #          y aplicar VUESTRAS DECISIONES (TSA que no responde, carpeta que ya existe,
    #          qué mostrar al terminar, fuente obligatoria u opcional).
    #          Si necesitas preguntar algo al usuario, usa input().
    # TODO 4 · Mostrar el resultado (carpeta, huella, hora del sello) o el error de forma comprensible,
    #          sin que salga un traceback.
    print("Pendiente: el botón todavía no hace nada. Completa ejecutar_sellado().")


if __name__ == "__main__":
    ejecutar_sellado()

### El prompt de tu IA

Copia este texto en tu IA, **completa «MIS DECISIONES»** (elige al menos dos) y adjunta los tres archivos
(`sellado_core.py`, `02_reto_colab_incompleto.py`; el formulario es solo para ti). Puedes ver su contenido con las celdas
`!cat` de abajo o descargarlos con la celda siguiente.

```text
OBJETIVO
Quiero que el «botón» de una pequeña herramienta funcione: relleno un formulario con un archivo, una fuente y unas
observaciones y una carpeta de destino; ejecuto una celda y se crea un paquete de evidencia.
Lo voy a ejecutar en Google Colab con %run (si hace falta preguntarme algo, que use input()).

LO QUE YA TENGO
- Un núcleo ya preparado y probado: sellado_core.py. Su función
  sellar_archivo(archivo, destino, fuente=..., observaciones=..., si_existe=..., reintentos=..., si_tsa_falla=...)
  conserva una copia exacta del original, calcula el SHA-256, prepara la petición, pide el sello a la TSA,
  comprueba que la huella sellada corresponde y escribe manifiesto.json.
  Devuelve un resultado con: carpeta, estado, sha256, hora_sello, emisor_tsa.
  Si algo falla lanza SelladoError (y ErrorTSA, que es un SelladoError, si la TSA no responde).
  Opciones: si_existe="error" o "numerar" · reintentos=0 o 1 · si_tsa_falla="abortar" o "pendiente".
- El archivo de partida: 02_reto_colab_incompleto.py. Ya lee el formulario con leer_formulario().
  Solo falta ejecutar_sellado().

ARCHIVOS
- sellado_core.py
- 02_reto_colab_incompleto.py

REQUISITOS
- Leer archivo, destino, fuente y observaciones con leer_formulario().
- Si falta el archivo o el destino: avisar con print y no hacer nada más.
- Llamar a sellar_archivo() con la fuente y las observaciones.
- Mostrar al terminar el resultado o un error comprensible (con print). No debe salir un traceback.

CRITERIOS DE ACEPTACIÓN (así sabré que funciona)
- El archivo de partida no cambia.
- No se sobrescribe un paquete anterior sin que yo lo decida.
- Se crea EVIDENCIA_<nombre> con original, peticion.tsq, sello.tsr y manifiesto.json.
- El manifiesto recoge mi fuente y mis observaciones.
- La huella sellada corresponde al original.
- Si algo falla, el mensaje se entiende y no queda un paquete a medias (salvo que yo elija «pendiente»).

LÍMITES
- No reimplementes el núcleo ni el protocolo de sellado. No modifiques sellado_core.py.
- Completa SOLO la función ejecutar_sellado(). No cambies el resto del archivo.
- Devuélveme el archivo 02_reto_colab_incompleto.py completo, listo para guardar.

MIS DECISIONES (elige al menos dos y borra las opciones que no uses)
- Si la TSA no responde:  [ ] abortar sin paquete   [ ] reintentar una vez y después abortar
                          [ ] crear un paquete marcado PENDIENTE_DE_SELLO
- Si la carpeta ya existe: [ ] impedirlo y avisar   [ ] crear _2, _3…   [ ] pedir otra carpeta de destino
- Al terminar:            [ ] mensaje   [ ] mensaje con la carpeta, la huella y la hora del sello
- Fuente o URL:           [ ] obligatoria   [ ] opcional, avisando si está vacía
```

Si no sabes por dónde empezar, o algo ha fallado, tienes `AYUDA_1.txt`, `AYUDA_2.txt` y `PROMPTS_DE_APOYO.txt`
en la carpeta `03_SELLADOR` (están pensados para la ventana de clase, pero la idea es la misma: los mensajes se muestran con `print`).

In [ ]:
ir_a("03_SELLADOR")
!cat sellado_core.py | head -60

In [ ]:
# Descarga a tu ordenador los dos archivos para adjuntarlos en tu IA
descargar("sellado_core.py", "02_reto_colab_incompleto.py")

## Por qué el prompt está construido así

**Objetivo · lo que ya tengo · requisitos · criterios · límites · decisiones.** Quien pide el código decide qué significa que «funcione».

## Antes de pedírselo a la IA: decidid cómo queréis que se comporte · 3 min

Hay cuatro decisiones: qué hacer si la TSA no responde · si la carpeta ya existe · qué mostrar al terminar · si la fuente es obligatoria.
Elegid al menos dos y **escribidlas en el prompt**.

## AHORA VOSOTROS · 15 min

**1.** Rellena el formulario (los campos de la ventana de clase).

In [ ]:
#@title Formulario del sellador (equivale a los campos de la ventana) { display-mode: "form" }
import json
ARCHIVO = "../02_INSPECTORAPP/captura_original.jpg" #@param {type:"string"}
DESTINO = "MIS_PAQUETES" #@param {type:"string"}
FUENTE = "https://hogarprueba.test/productos/orion-3" #@param {type:"string"}
OBSERVACIONES = "Prueba del taller en Colab." #@param {type:"string"}
ir_a("03_SELLADOR")
json.dump({"archivo": ARCHIVO, "destino": DESTINO, "fuente": FUENTE, "observaciones": OBSERVACIONES},
          open("formulario.json", "w", encoding="utf-8"), ensure_ascii=False)
print("Formulario guardado. (Si cambias un campo, vuelve a ejecutar esta celda.)")

**2.** Pega el código que te dé la IA **debajo de la primera línea** de la celda siguiente (deja `%%writefile mi_sellador.py` arriba).
Debe ser el archivo completo: `leer_formulario()` y `ejecutar_sellado()` completos.

In [ ]:
ir_a("03_SELLADOR")

In [ ]:
%%writefile mi_sellador.py
# Pega aquí, debajo de esta línea, el código que te ha dado la IA

**3.** «Pulsa el botón»: esta celda ejecuta tu versión. Si te pide datos, aparecerá un cuadro de texto.

In [ ]:
ir_a("03_SELLADOR")
%run mi_sellador.py

**¿Ha funcionado?** Comprueba tus criterios de aceptación, uno por uno (no basta con que no haya error).
Si ha fallado: copia el **error completo** y vuelve a la IA, como en el Bloque 1.

> **Si la TSA no responde desde Colab:** puede ocurrir (la red de Google no es la de tu equipo ni la del aula).
> Es justo lo que has tenido que decidir en «si la TSA no responde». Para seguir con el resto del bloque tienes un paquete
> de contingencia preparado; las celdas de comprobación lo usan automáticamente si no has creado uno.

## Resultado · así queda cuando funciona

Un paquete `EVIDENCIA_captura_original` con cuatro archivos. Míralo:

In [ ]:
ir_a("03_SELLADOR")
paquete = ultimo_paquete()
print("Tu paquete:", paquete if paquete else "(todavía no has creado ninguno)")
if paquete:
    !ls "{paquete}"

## SELLADOR ≠ VERIFICADOR

Quien produce el paquete **no se certifica a sí mismo**. Necesitamos algo independiente que lo compruebe.

## `COMPROBAR_PAQUETE` · 2 min

*(En clase: arrastrar la carpeta `EVIDENCIA_…` sobre `COMPROBAR_PAQUETE.bat`.)* No se fía del manifiesto: vuelve a calcular la huella
del original, lee el sello y comprueba que todo encaja. **No** comprueba la firma ni la cadena de confianza (eso lo hace `/verificador`).

In [ ]:
ir_a("03_SELLADOR")
paquete = ultimo_paquete()
if paquete is None:
    paquete = BASE / "03_SELLADOR" / "PAQUETE_CONTINGENCIA" / "EVIDENCIA_captura_original_DEMO"
    print("No tienes paquete propio: se comprueba el de contingencia (TSA de demostración, NO cualificada).\n")
!python comprobar_paquete.py "{paquete}"

**Que el programa termine sin error ≠ paquete validado.** Para eso tiene que pasar la comprobación independiente.

## Prueba negativa a mano · ¿el sello acepta cualquier archivo? · 3 min

**Predicción.** Vamos a copiar `original.jpg` de tu paquete, **pintarle un punto** y comprobar la copia con **el mismo** `sello.tsr`.
¿Qué cambiará? el sello · su firma · la huella del archivo · la correspondencia

*(En clase: Paint. Aquí: ADAPTADA, una celda con Pillow que pinta un punto rojo.)*

In [ ]:
from PIL import Image, ImageDraw
ir_a("03_SELLADOR")
paquete = ultimo_paquete() or (BASE / "03_SELLADOR" / "PAQUETE_CONTINGENCIA" / "EVIDENCIA_captura_original_DEMO")
img = Image.open(paquete / "original.jpg").convert("RGB")
ImageDraw.Draw(img).ellipse((20, 20, 30, 30), fill=(255, 0, 0))     # el punto
img.save("copia_modificada.jpg")
print("Creada copia_modificada.jpg (el original del paquete no se ha tocado).")

Ahora, **la copia + el mismo sello**: (en clase: arrastrar la copia y el `sello.tsr` sobre `3_VERIFICAR_SELLO.bat`)

In [ ]:
ir_a("03_SELLADOR")
paquete = ultimo_paquete() or (BASE / "03_SELLADOR" / "PAQUETE_CONTINGENCIA" / "EVIDENCIA_captura_original_DEMO")
!python ../02_INSPECTORAPP/verificar_sello.py copia_modificada.jpg "{paquete}/sello.tsr"

Compara con el original: ¿qué ha cambiado y qué no? **El mismo sello. Otro archivo.**

También en la web (`/verificador`, igual que en clase, en tu navegador):

In [ ]:
ir_a("03_SELLADOR")
paquete = ultimo_paquete() or (BASE / "03_SELLADOR" / "PAQUETE_CONTINGENCIA" / "EVIDENCIA_captura_original_DEMO")
descargar("copia_modificada.jpg", str(paquete / "original.jpg"), str(paquete / "sello.tsr"))

## Os lleváis un punto de partida

Hoy: **archivo → sello → paquete**. En otro proyecto podría adaptarse a PDF, HTML, JSON, descargas, capturas, lotes, scraping…
*No es un producto universal terminado: es una base reutilizable y adaptable.*

> **Recuerda:** un paquete creado aquí es un ejercicio. No equivale a una captura hecha desde un equipo institucional.

---
# J · CONSUMOKU

**Diez inspectores. Un encargo. Una ganadora.** Sois el jurado. Ganará la primera candidatura **válida** recibida.

## El encargo y el método

El docente te dirá dónde descargar `CONSUMOKU_ALUMNADO_FINAL.zip` al empezar el reto (en clase: se descomprime dentro de `04_CONSUMOKU`).
Contiene el caso y las reglas, el manifiesto anónimo, las diez imágenes, los sellos y la hoja del jurado.

> **ADAPTADA.** Descarga el ZIP en tu ordenador y súbelo aquí con la celda siguiente
> (o arrástralo al panel 📁 de la izquierda, a la carpeta de trabajo).

**El método del reto: un embudo.**
**RELACIONA → DESCARTA RÁPIDO → REVISA EL ARCHIVO → VERIFICA SOLO LAS SUPERVIVIENTES → DECIDE.**
No verifiques sellos de candidaturas que ya has descartado por algo más barato.

In [ ]:
import zipfile, shutil
from pathlib import Path
nombre_zip = "CONSUMOKU_ALUMNADO_FINAL.zip"
candidatos = [Path.cwd() / nombre_zip, TRABAJO / nombre_zip, BASE / "04_CONSUMOKU" / nombre_zip]
zip_consumoku = next((c for c in candidatos if c.exists()), None)
if zip_consumoku is None:
    try:
        from google.colab import files
        print("Elige CONSUMOKU_ALUMNADO_FINAL.zip en tu ordenador...")
        files.upload()
        zip_consumoku = Path.cwd() / nombre_zip
    except ImportError:
        print("No se encuentra el ZIP: colócalo en la carpeta de trabajo y vuelve a ejecutar la celda.")
if zip_consumoku and zip_consumoku.exists():
    destino = BASE / "04_CONSUMOKU"
    with zipfile.ZipFile(zip_consumoku) as z:
        z.extractall(destino)
    print("Material descomprimido en:", destino.relative_to(TRABAJO))
    for p in sorted(destino.iterdir()):
        print("  ", p.name)

## El encargo · léelo con atención

Abre `00_CASO_Y_REGLAS_FINAL.pdf` (en el panel 📁 → `BLOQUE4/04_CONSUMOKU`, o descárgalo) y anota: **qué se buscaba, dónde, cuándo, con qué canal y con qué aplicación**.

In [ ]:
ir_a("04_CONSUMOKU")
descargar("00_CASO_Y_REGLAS_FINAL.pdf")

## Trabajo del jurado · 22 min (con checkpoint hacia el minuto 8–9)

Trabajad por parejas. Las herramientas son las mismas que habéis usado: metadatos, huellas, sellos, y las apps web del curso.

**Checkpoint (minuto 8–9):** ¿cuántas candidaturas siguen vivas? ¿Estáis verificando sellos de candidaturas ya descartadas?

### 1 · El manifiesto anónimo

In [ ]:
ir_a("04_CONSUMOKU")
ver_excel("01_MANIFIESTO_ANONIMO_FINAL.xlsx")

### 2 · La hoja del jurado

In [ ]:
ver_excel("04_HOJA_DEL_JURADO.xlsx")

### 3 · Las imágenes y sus metadatos
*(Mira las imágenes con `display`; en clase se abrían desde la carpeta.)*

In [ ]:
ir_a("04_CONSUMOKU")
!python ../02_INSPECTORAPP/ver_metadatos.py 02_IMAGENES_ANONIMAS

In [ ]:
# Cambia el número para ver otra imagen
display(VerImagen("02_IMAGENES_ANONIMAS/archivo_01.jpg", width=720))

### 4 · Huellas de todas las imágenes

In [ ]:
ir_a("04_CONSUMOKU")
!python ../02_INSPECTORAPP/comparar_hashes.py 02_IMAGENES_ANONIMAS

### 5 · Verificar un sello (solo para las supervivientes)

Elige el archivo y su sello y **cambia los dos nombres**. El ejemplo de abajo solo muestra cómo se usa:
**qué sello acompaña a cada archivo lo tienes que establecer tú** con el manifiesto (la huella declarada, el nombre del sello…),
no lo da el ejemplo.

In [ ]:
ir_a("04_CONSUMOKU")
ARCHIVO_A_VERIFICAR = "02_IMAGENES_ANONIMAS/archivo_01.jpg"
SELLO = "03_SELLOS/INS-01_sello.tsr"
!python ../02_INSPECTORAPP/verificar_sello.py "{ARCHIVO_A_VERIFICAR}" "{SELLO}"

También puedes usar `/verificador`, `/hash` y `/exif` en la web (descargando antes los archivos con `descargar(...)`).

### Tu fallo del jurado

Anota: **la ganadora**, la **segunda válida** si la hay, y las **descartadas con su motivo en una frase**.
El formulario anónimo de fallo está en la web del curso, igual que en clase.

---
# K · Resolución y cierre

La resolución se hace **en común**: una candidatura tras otra, con el motivo y la decisión. *(No hay nada que ejecutar.)*
Cuando el docente la muestre, compara con tu fallo.

## Lo que hoy hemos aprendido a no confundir

- **METADATO ≠ DATO CERTIFICADO**
- **COMPATIBLE ≠ DEMOSTRADO**
- **IP OBSERVADA ≠ PERSONA IDENTIFICADA**
- **INTEGRIDAD ≠ VERACIDAD**
- **SELLO DE TIEMPO ≠ FECHA DE CREACIÓN**
- **QUE TERMINE SIN ERROR ≠ PAQUETE VALIDADO**
- **VALIDEZ TÉCNICA ≠ SUFICIENCIA**
- **VERIFICAR EL MECANISMO ≠ DEMOSTRAR EL HECHO**

*El bloque no sustituye un protocolo formal ni la valoración jurídica de una prueba.*

**Fuentes:** Reglamento (UE) 910/2014 (eIDAS) · RFC 3161 · FIPS 180-4.

✅ **Bloque 4 completado.** Sé qué recoger.

---
# Solo si te has bloqueado

**Restaurar los materiales originales** (por ejemplo, si has estropeado un script).
Atención: borra también tus paquetes, `mi_sellador.py`, `mi_copia.jpg` y el material de CONSUMOKU que hayas descomprimido.

In [ ]:
import shutil
shutil.rmtree(BASE, ignore_errors=True)
with zipfile.ZipFile(TRABAJO / "materiales_b4.zip") as z:
    z.extractall(TRABAJO)
print("Materiales restaurados.")
ir_a("")

**Solución de referencia del sellador** (no es «la respuesta correcta», solo una referencia). Es la versión Colab de
`03_solucion_referencia.py`: las cuatro decisiones están arriba; cambia un valor para probar otra opción.
Esta celda la guarda; la ejecuta la celda siguiente.

In [ ]:
ir_a("03_SELLADOR")

In [ ]:
%%writefile 03_solucion_referencia_colab.py
"""TALLER DEL SELLADOR · SOLUCIÓN DE REFERENCIA (versión Colab).

Es la misma lógica que 03_solucion_referencia.py (ventana), con mensajes por print/input.
Las cuatro decisiones del mini-reto están arriba: cambia el valor para probar otra opción.
"""
import json
from pathlib import Path

from sellado_core import ErrorTSA, SelladoError, sellar_archivo

# ---- DECISIONES (mini-reto del prompt) -------------------------------------------------
SI_TSA_NO_RESPONDE = "abortar"        # "abortar" · "reintentar" (una vez y abortar) · "pendiente"
SI_CARPETA_EXISTE = "impedir"         # "impedir" · "numerar" (_2, _3…) · "otro_destino" (pide otra carpeta)
AL_TERMINAR = "mensaje_y_ruta"        # "mensaje" · "mensaje_y_ruta"
FUENTE_REQUERIDA = "opcional_con_aviso"   # "obligatoria" · "opcional_con_aviso"
# ----------------------------------------------------------------------------------------


def leer_formulario():
    datos = json.loads(Path("formulario.json").read_text(encoding="utf-8"))
    return datos["archivo"], datos["destino"], datos["fuente"], datos["observaciones"]


def ejecutar_sellado():
    # 1 · leer
    archivo, destino, fuente, observaciones = leer_formulario()
    fuente, observaciones = fuente.strip(), observaciones.strip()
    # 2 · validar
    if not archivo:
        print("AVISO · Falta el archivo: elige primero el archivo que quieres conservar.")
        return
    if not destino:
        print("AVISO · Falta el destino: indica la carpeta donde se creará el paquete.")
        return
    if not fuente:
        if FUENTE_REQUERIDA == "obligatoria":
            print("AVISO · Falta la fuente: indica la fuente o URL de la que procede el archivo.")
            return
        if input("No has indicado fuente o URL. ¿Crear el paquete igualmente? (s/n): ").strip().lower() not in ("s", "si", "sí"):
            print("Cancelado. No se ha creado ningún paquete.")
            return
    # 3 · llamar al núcleo con las decisiones
    opciones = {
        "fuente": fuente,
        "observaciones": observaciones,
        "si_existe": "numerar" if SI_CARPETA_EXISTE == "numerar" else "error",
        "reintentos": 1 if SI_TSA_NO_RESPONDE == "reintentar" else 0,
        "si_tsa_falla": "pendiente" if SI_TSA_NO_RESPONDE == "pendiente" else "abortar",
    }
    print("Creando la petición y contactando con la TSA…")
    try:
        try:
            resultado = sellar_archivo(archivo, destino, **opciones)
        except SelladoError as exc:
            if SI_CARPETA_EXISTE == "otro_destino" and str(exc).startswith("Ya existe la carpeta"):
                nuevo = input("Ya existe un paquete ahí. Escribe otra carpeta de destino (Enter para cancelar): ").strip()
                if not nuevo:
                    print("Cancelado. No se ha creado ningún paquete.")
                    return
                resultado = sellar_archivo(archivo, nuevo, **opciones)
            else:
                raise
    except ErrorTSA as exc:
        print(f"ERROR · La TSA no ha respondido.\n{exc}\n\nNo se ha creado ningún paquete. El archivo de partida no se ha tocado.")
    except SelladoError as exc:
        print(f"ERROR · No se pudo crear el paquete.\n{exc}\n\nEl archivo de partida no se ha tocado.")
    except Exception as exc:  # error inesperado: se muestra y la celda termina sin traceback
        print("ERROR · Error inesperado:", repr(exc))
    else:
        # 4 · mostrar el resultado
        if resultado.estado == "PENDIENTE_DE_SELLO":
            print("AVISO · La TSA no respondió. Paquete creado y marcado PENDIENTE DE SELLO:")
            print(f"  {resultado.carpeta}\nNo tiene garantía temporal externa.")
        else:
            print("Paquete creado. La huella sellada corresponde al original.")
            if AL_TERMINAR == "mensaje_y_ruta":
                print(f"  Carpeta: {resultado.carpeta}\n  SHA-256: {resultado.sha256}\n  Hora del sello: {resultado.hora_sello}")
            print("Compruébalo con comprobar_paquete.py")


if __name__ == "__main__":
    ejecutar_sellado()

In [ ]:
import json
ir_a("03_SELLADOR")
if not Path("formulario.json").exists():
    print("No hay formulario: se usan los datos por defecto (captura_original.jpg).\n")
    json.dump({"archivo": "../02_INSPECTORAPP/captura_original.jpg", "destino": "MIS_PAQUETES",
               "fuente": "https://hogarprueba.test/productos/orion-3", "observaciones": "Prueba con la referencia."},
              open("formulario.json", "w", encoding="utf-8"), ensure_ascii=False)
%run 03_solucion_referencia_colab.py

**Paquete de contingencia** (muestra técnica, TSA de demostración **no cualificada**): ya lo tienes en
`03_SELLADOR/PAQUETE_CONTINGENCIA/EVIDENCIA_captura_original_DEMO`. Sirve para seguir con `COMPROBAR_PAQUETE` y la prueba negativa
si la TSA no responde.

**Prompts de apoyo** (para un error o para comparar con la referencia):

In [ ]:
!cat "{BASE}/03_SELLADOR/PROMPTS_DE_APOYO.txt"